# Analysis - DINOv2 + linear probe

**Owner:** suprch4rg3d   |   **Artifacts:** `results/dinov2/`

This notebook shows how well DINOv2 detects fake images. It reads only the
metrics saved in `results/` (no model runs, no re-training) and draws the
figures inline. Every number below comes from a metrics JSON produced on
the L4 run.

**How to use:** make sure `results/` is present (it ships in the repo and is
also backed up to Drive), then run the code cells in order.


## 0. The storyline

Three questions guide this analysis:

1. **Does it work at all?** A frozen DINOv2 ViT-B/14 (embeddings at 224x224,
   no fine-tuning) plus one logistic layer should separate GenImage SD real
   from SD fake on its own train distribution.
2. **Does the skill transfer?** Training sees only SD v1.4 images. The
   question is whether the probe still separates real from fake on
   Midjourney, ADM and BigGAN - generators it never saw.
3. **Is it the pixels or the bytes?** GenImage SD fakes are larger files
   than reals (median ~458 KB vs ~127 KB). A model that only "detects file
   size" has learned nothing about images. We test this two independent
   ways: a file-size threshold probe and a re-compression probe that
   removes the size signal but keeps the pixels.


## Palette

All figures use the same colors:

- **DINOv2 + probe** - violet `#7c3aed`
- **ResNet baseline** - slate `#94a3b8`
- **ensemble / fused** - pink `#ec4899`
- **recompressed (size removed)** - amber `#f59e0b`
- **chance line (0.50)** - red `#ef4444`


## 1. In-domain accuracy: does it work at all?

`extract-dinov2` embeds every image with the frozen model (`dinov2_vitb14`,
224x224) and `train-linear-probe` fits one logistic layer on the train
features. A linear head is weak by design: if separation appears, it comes
from the features, not from a strong classifier.

| split   | accuracy |
|---------|----------|
| train   | **0.8873** |
| heldout | **0.8746** |

(For context, the project's fine-tuned ResNet baseline reaches 0.9703 /
0.9666 on the same splits.)

**Reading:** heldout is close to train (gap of 1.3 points). The probe is
not overfitting. The frozen features separate SD real from SD fake.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Locate the repo root no matter where this notebook is opened from
# (repo root, notebooks/, or Colab's /content/fauxtography).
def _find_results(start: Path) -> Path:
    for d in (start, *start.parents):
        if (d / "results").is_dir():
            return d / "results"
    raise FileNotFoundError("results/ not found - open the notebook from the repo root")

results = _find_results(Path.cwd())
dinov2 = results / "dinov2"

def acc(*parts):
    """Read accuracy from a metrics.json under results/."""
    parts = (*parts,)
    p = results.joinpath(*parts)
    if p.is_dir():
        p = p / "metrics.json"
    return json.loads(p.read_text())["accuracy"]

# In-domain table
pd.DataFrame(
    {"split": ["train", "heldout"],
     "accuracy": [acc("dinov2", "metrics.json"), acc("dinov2", "heldout_metrics.json")]}
).style.hide(axis="index")


## 2. In-domain confusion

The confusion matrices are committed as SVGs:

- `results/dinov2/confusion_matrix.svg` (train)
- `results/dinov2/heldout_confusion_matrix.svg` (heldout)

Heldout errors go both ways: the model misses some fake images and
false-flags some real ones, with no strong imbalance. The raw
`y_true/y_pred.npy` files are committed if you want to relabel the errors
yourself.


## 3. Does the skill transfer? Cross-generator generalization

`eval-cross` takes the **same trained probe - zero retraining** - and
scores it on 2,000-image sets (1,000 real + 1,000 fake) from three
generators the model never saw. The GenImage paper trains a ResNet per
generator; here we use one probe on all generators.

| generator | DINOv2 probe | ResNet baseline |
|-----------|-------------:|----------------:|
| Midjourney | **0.8205** | 0.5945 |
| ADM        | 0.5100       | 0.4910 |
| BigGAN     | 0.5545       | 0.4950 |

All sets are balanced (1,000 real / 1,000 fake each), so 0.50 is chance.

**Result:**

- **Midjourney transfers well.** 0.82 with no Midjourney images in
  training, 23 points above the ResNet's 0.59.
- **ADM and BigGAN score near chance for both models.** DINOv2 0.51 / 0.55;
  ResNet 0.49 / 0.50. Detection does not transfer across generator
  families.
- DINOv2 beats the ResNet on all three (up to +0.23 on Midjourney), but
  both models fail on ADM and BigGAN.


In [ ]:
# Figure 1 - The generator cliff: DINOv2 vs ResNet across all populations
groups = ["train", "heldout", "Midjourney", "ADM", "BigGAN"]
din = [
    acc("dinov2", "metrics.json"),
    acc("dinov2", "heldout_metrics.json"),
    acc("dinov2", "cross", "Midjourney", "metrics.json"),
    acc("dinov2", "cross", "ADM", "metrics.json"),
    acc("dinov2", "cross", "BigGAN", "metrics.json"),
]
base = [
    acc("baseline", "metrics.json"),
    acc("baseline", "heldout_metrics.json"),
    acc("baseline", "cross", "Midjourney", "metrics.json"),
    acc("baseline", "cross", "ADM", "metrics.json"),
    acc("baseline", "cross", "BigGAN", "metrics.json"),
]

x = np.arange(len(groups))
w = 0.36
fig, ax = plt.subplots(figsize=(10, 5.2), dpi=200)
b1 = ax.bar(x - w/2, base, w, label="ResNet baseline", color="#94a3b8")
b2 = ax.bar(x + w/2, din, w, label="DINOv2 + probe", color="#7c3aed")
ax.axhline(0.5, color="#ef4444", ls="--", lw=1.2)
ax.text(4.42, 0.51, "0.50 = chance", color="#ef4444", fontsize=9)
for bars in (b1, b2):
    for rect in bars:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.008,
                f"{h:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.axvline(1.5, color="black", lw=0.6, alpha=0.4)
ax.text(0.75, 0.98, "SEEN\n(SD v1.4)", ha="center", va="top", fontsize=8, color="#555")
ax.text(3.25, 0.98, "UNSEEN GENERATORS", ha="center", va="top", fontsize=8, color="#555")
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.06)
ax.legend(loc="upper left", frameon=False)
ax.set_title("Cross-generator detection: DINOv2 probe vs ResNet baseline "
             "(same probe, zero retraining)", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


## 4. Is the model using the pixels or the file size?

GenImage SD fakes are larger files than reals. If a model reads byte
counts instead of image content, its accuracy is not meaningful. Two
independent probes test this.

### 4a. A classifier that only uses file size

`probe-file-size` fits one threshold on file size and reports what a model
that used *only the bytes* would score (`results/file_size/`):

| set      | n       | accuracy |
|----------|--------:|---------:|
| train    | 14,000  | 0.9896   |
| test     | 12,000  | 0.9873   |
| Midjourney | 2,000 | 0.9885   |
| ADM      | 2,000   | 0.4885   |
| BigGAN   | 2,000   | 0.4865   |

Threshold: ~258 KB. The size distribution is in
`results/file_size/size_histogram.svg`.

**Reading 4a:** on its own, file size separates the training pool well
(0.99). But it is anti-correlated with the cross-generator sets: ADM and
BigGAN fakes are smaller than their reals, so a size-based model drops to
0.49 there. A detector that depends on file size would pass on SD and fail
on ADM/BigGAN - the same pattern seen in section 3. So this is a real
concern we must rule out.


### 4b. Re-compression probe: remove the size signal, keep the pixels

`probe-compression` re-compresses every held-out image toward the real
class's median file size (fake sizes drop from ~458 KB to ~95 KB, close to
real ~87 KB), then re-scores the *unchanged* probe. The pixel content
stays; the size signal is removed. Result (`results/dinov2/compression/`,
n = 12,000):

| condition               | median fake size | accuracy |
|------------------------|-----------------:|---------:|
| original heldout       | ~458 KB          | **0.8746** |
| re-compressed to real | ~95 KB           | **0.8678** |

**Reading 4b:** accuracy barely changes (0.875 -> 0.868). Even without the
size difference, the probe still separates real from fake. The model
reads pixels, not bytes. The size signal exists in the data (4a), but the
model does not rely on it. The ~0.87 on compressed images is the
shortcut-free accuracy.


In [ ]:
# Figure 2 - Compression probe: destroy the size signal, keep the pixels
orig = acc("dinov2", "heldout_metrics.json")
c = json.loads((dinov2 / "compression" / "metrics.json").read_text())
recom = c["accuracy"]
before, after = c["median_bytes_before"], c["median_bytes_after"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 4.2), dpi=200,
                               gridspec_kw={"width_ratios": [1, 1.25]})

ax1.bar(["original\nheldout", "recompressed\nto real size"],
        [orig, recom], color=["#7c3aed", "#f59e0b"], width=0.55)
for i, h in enumerate((orig, recom)):
    ax1.text(i, h + 0.006, f"{h:.3f}", ha="center", va="bottom", fontsize=10)
ax1.set_ylim(0, 1.05)
ax1.set_ylabel("accuracy")
ax1.axhline(0.5, color="#ef4444", ls="--", lw=1)
ax1.set_title("Accuracy survives re-compression", fontsize=10)
ax1.spines[["top", "right"]].set_visible(False)

ax2.bar(["real", "fake"],
        [before["real"]/1024, before["fake"]/1024],
        width=0.4, color="#94a3b8", label="original")
ax2.bar([i + 0.42 for i in range(2)],
        [after["real"]/1024, after["fake"]/1024],
        width=0.4, color="#f59e0b", label="recompressed")
ax2.set_xticks(range(2)); ax2.set_xticklabels(["real", "fake"])
ax2.set_ylabel("median file size (KB)")
for i, (b, a) in enumerate(zip(
        [before["real"]/1024, before["fake"]/1024],
        [after["real"]/1024, after["fake"]/1024])):
    ax2.text(i, b + 6, f"{b:.0f}", ha="center", fontsize=9)
    ax2.text(i + 0.42, a + 6, f"{a:.0f}", ha="center", fontsize=9, color="#f59e0b")
ax2.set_ylim(0, 540)
ax2.legend(frameon=False)
ax2.set_title("Size gap destroyed", fontsize=10)
ax2.spines[["top", "right"]].set_visible(False)

fig.suptitle("Compression probe: not a file-size cheat", fontsize=12)
plt.show()


## 5. What does the feature space look like?

`visualize-features` runs PCA then t-SNE on the cached DINOv2 features and
renders a scatter colored by generator and label. The committed figure is
at `results/dinov2/embedding/embedding.svg` (8,000 points, 8 classes - 3
cross generators + train, each real/fake).

**Reading:** each generator occupies its own region of feature space. This
is consistent with the cross-generator results: separation works inside
one cluster (in-domain 0.87, Midjourney 0.82) but fails across clusters
(ADM, BigGAN). The probe's decision boundary is tuned to SD-style cues,
and each new generator sits in a different region.


In [ ]:
# Show the committed feature embedding (generated by visualize-features)
from IPython.display import SVG
emb = dinov2 / "embedding" / "embedding.svg"
if emb.exists():
    meta = json.loads((dinov2 / "embedding" / "embedding_classes.json").read_text())
    display(SVG(filename=str(emb)))
    print(f"DINOv2 feature space (PCA->t-SNE, {meta['n_points']} points)")
else:
    print("no embedding on disk yet")


## 6. Does combining with the baseline help?

`analyze-fusion` takes a majority vote of the DINOv2 probe and the ResNet
baseline (CLIP joins once its prompt sweep lands; the summary tolerates a
missing CLIP). `results/fusion/summary.json`:

| split   | fused | baseline | dinov2 | agreement |
|---------|------:|---------:|-------:|----------:|
| internal | 0.9387 | 0.9703 | 0.8873 | 0.8730 |
| heldout | 0.9288 | 0.9666 | 0.8746 | 0.8597 |

**Reading:** the two models agree on ~86-87% of samples and disagree on a
useful 13-14%. The vote reaches 0.929 on heldout - above DINOv2 alone
(0.875) and close to the baseline (0.967). DINOv2's main value is
complementarity: it classifies correctly some images the ResNet gets
wrong.


In [ ]:
# Figure 3 - The ensemble: baseline + DINOv2 beat either alone
rows = json.loads((results / "fusion" / "summary.json").read_text())
splits = [r["split"] for r in rows]
ba = [r["model_accuracies"]["baseline"] for r in rows]
di = [r["model_accuracies"]["dinov2"] for r in rows]
fu = [r["fused_accuracy"] for r in rows]

x = np.arange(len(splits)); w = 0.25
fig, ax = plt.subplots(figsize=(7.5, 4.6), dpi=200)
b1 = ax.bar(x - w, ba, w, label="ResNet baseline", color="#94a3b8")
b2 = ax.bar(x, di, w, label="DINOv2 + probe", color="#7c3aed")
b3 = ax.bar(x + w, fu, w, label="ensemble (vote)", color="#ec4899")
for bars in (b1, b2, b3):
    for rect in bars:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.004,
                f"{h:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.12)
ax.legend(frameon=False, loc="upper right")
ax.set_title("Ensemble of baseline + DINOv2 beats either model alone", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


## 7. Verdict

DINOv2 + linear probe **works on its own distribution** (0.875 heldout),
**is better than the ResNet baseline on Midjourney** (0.82 vs 0.59, no
generator-specific training), and **does not generalize to every
generator** (ADM 0.51, BigGAN 0.55, though it still edges the baseline
there). It is **not a file-size shortcut** (compression: 0.868). Its main
use is as a **second detector**: the ensemble reaches 0.93 heldout.

**One sentence for the slides:** *A frozen DINOv2 plus one linear layer
detects SD fakes and Midjourney fakes without seeing them, does not use
file size - but per-generator style drift limits it, and its value is in
lifting the ensemble above either model alone.*

**Caveats:** the cross-generator evaluation uses one probe with no
per-generator training (the paper trains a ResNet per generator), and the
DINOv2 training slice is smaller than the paper's - so the 0.87 headline
is not directly comparable to the paper's ResNet numbers. The CLIP column
is pending Demetra's prompt sweep.
